# Identification when the graph is not a DAG until you unroll it

`CausalGraph` carries a `feedback` flag whose only honest use is to say "the static summary
graph hides treatment–outcome feedback, so static adjustment is not enough". That flag is a
refusal. This is the answer to it: unroll the system over the periods you plan to analyse
and ask the ordinary graphical questions of the ordinary DAG that comes out.

In [ ]:
from axiom.core import D, Param, dimensionless
from axiom.dynamics import Variable, parse_system
from axiom.identify import (
    CausalGraph, SequentialPlan, identify, sequential_backdoor_admissible, sequential_plan,
    unrolled_graph,
)

NONE = dimensionless()
system = parse_system(
    """
    outcome = beta * dose + rho * outcome[t-1] + kappa * frailty
    dose    = phi * outcome[t-1] + protocol
    """,
    variables=(
        Variable(name="outcome", dimension=D.outcome),
        Variable(name="dose", dimension=D.currency),
        Variable(name="protocol", dimension=D.currency, role="exogenous"),
        Variable(name="frailty", dimension=NONE, role="exogenous", observed=False),
    ),
    parameters=(
        Param(name="beta", dimension=D.outcome / D.currency),
        Param(name="rho", dimension=NONE),
        Param(name="kappa", dimension=D.outcome),
        Param(name="phi", dimension=D.currency / D.outcome),
    ),
    name="dose-responds-to-outcome",
)

## The unrolled graph

One node per variable per period; a variable declared `observed=False` is unmeasured at
every period. The result is an ordinary `CausalGraph`, so everything in `axiom.identify`
already works on it.

In [ ]:
graph = unrolled_graph(system, periods=3)
print(graph.nodes)
print(graph.to_text())
print("unmeasured:", graph.unmeasured)
print("topological order:", graph.topological_order())

## Why one static adjustment set cannot work

`outcome.t1` is a **mediator** of `dose.t0` and a **confounder** of `dose.t2`. The static
back-door criterion demands adjusting for it and forbids adjusting for it, from the same
graph. That is not a defect of the criterion; it is why the g-formula exists.

In [ ]:
print("descendant of dose.t0:", "outcome.t1" in graph.descendants("dose.t0"))
print("parent of dose.t2:", "outcome.t1" in graph.parents("dose.t2"))
static = identify(graph, "dose.t1", "outcome.t2")
print("static verdict for one stage:", static.verdict.status, static.route, static.adjustment_set)

## The sequential plan

`sequential_plan` runs the sequential back-door criterion stage by stage: at stage $k$ the
covariates must be measured non-descendants of $A_k \ldots A_n$, and $A_k$ must be
d-separated from the outcome given the history in the graph with all edges *out of*
$A_k \ldots A_n$ deleted.

Positivity is not a graphical property, so a plan that passes comes back **downgraded**,
not identified, carrying positivity as a named unverified assumption.

In [ ]:
plan: SequentialPlan = sequential_plan(graph, ["dose.t0", "dose.t1", "dose.t2"], "outcome.t2")
print("status:", plan.verdict.status, "| licensed:", plan.licensed)
for stage, extra, full in zip(plan.stages, plan.adjustments, plan.conditioning_at):
    print(f"  {stage}: introduces {extra or '()'}, conditions on {full or '()'}")
print("static adjustment fails:", plan.static_adjustment_fails)
print("assumptions:", [(a.name, a.state) for a in plan.verdict.assumptions])

In [ ]:
# Conditioning on a descendant of a later treatment is refused, with the reason.
ok, why = sequential_backdoor_admissible(
    graph, ["dose.t0", "dose.t1"], "outcome.t2", [("outcome.t1",), ()]
)
print(ok, "->", why)

## When no plan exists

Make the confounder persist *and* drive the dose, and no adjustment set closes the back
door at any stage. The verdict is blocked, and it names the stage and what would fix it.

In [ ]:
confounded = parse_system(
    """
    outcome = beta * dose + rho * outcome[t-1] + kappa * frailty
    dose    = phi * outcome[t-1] + protocol + omega * frailty
    frailty = persist * frailty[t-1]
    """,
    variables=(
        Variable(name="outcome", dimension=D.outcome),
        Variable(name="dose", dimension=D.currency),
        Variable(name="protocol", dimension=D.currency, role="exogenous"),
        Variable(name="frailty", dimension=NONE, observed=False),
    ),
    parameters=(
        Param(name="beta", dimension=D.outcome / D.currency),
        Param(name="rho", dimension=NONE),
        Param(name="kappa", dimension=D.outcome),
        Param(name="phi", dimension=D.currency / D.outcome),
        Param(name="omega", dimension=D.currency),
        Param(name="persist", dimension=NONE),
    ),
    name="unmeasured-time-varying-confounder",
)
blocked = sequential_plan(unrolled_graph(confounded, periods=3), ["dose.t0", "dose.t1", "dose.t2"], "outcome.t2")
print(blocked.verdict.status)
print(blocked.verdict.reason)

In [ ]:
# Measure the confounder and the same system becomes licensed.
measured = confounded.model_copy(update={
    "variables": tuple(
        v.model_copy(update={"observed": True}) if v.name == "frailty" else v
        for v in confounded.variables
    )
})
plan2 = sequential_plan(unrolled_graph(measured, periods=3), ["dose.t0", "dose.t1", "dose.t2"], "outcome.t2")
print(plan2.verdict.status, plan2.adjustments)

## Simultaneity

A contemporaneous cycle is not a DAG and never will be. Its *reduced form* is, and that is
what `unrolled_graph` emits by default. Asking for the structural edges of a cyclic system
raises, naming the cycle — the honest outcome, since no DAG algorithm can answer a question
about it.

In [ ]:
from axiom.identify import GraphError

market = parse_system(
    """
    quantity = a - b * price
    price    = d + e * quantity + cost
    """,
    variables=(
        Variable(name="quantity", dimension=D.outcome),
        Variable(name="price", dimension=D.currency),
        Variable(name="cost", dimension=D.currency, role="exogenous"),
    ),
    parameters=(
        Param(name="a", dimension=D.outcome),
        Param(name="b", dimension=D.outcome / D.currency),
        Param(name="d", dimension=D.currency),
        Param(name="e", dimension=D.currency / D.outcome),
    ),
    name="market",
)
reduced = unrolled_graph(market, periods=2)
print("reduced:", reduced.to_text())
try:
    unrolled_graph(market, periods=2, reduced=False)
except GraphError as e:
    print("structural ->", e)